In [ ]:
import json
import os

import earthaccess
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pandarallel import pandarallel

In [ ]:
def specialClip(sj):
    left = gpd.GeoSeries(sj.geometry)
    right = gpd.GeoSeries(sj.geom)
    geometry_clip = left.clip(right)
    return geometry_clip

### Make table of intersecting USGS gauges and SWOT tiles

#### USGS gauges

In [ ]:
# Table of all gauges with NHDPlus HR reaches mapped
nhd_sites = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations_with_stage_and_nhd_metadata.parquet')

In [ ]:
# data_path = '/nas/cee-water/cjgleason/fiona/USGS_gauges/sites/'

# gauges_w_data = []

# for item in os.listdir(data_path):
#     gauges_w_data.append(item[:-8])

In [ ]:
# # Subset gauges to those with available stage data
# nhd_sites = nhd_sites[nhd_sites['site_id'].isin(gauges_w_data)].reset_index()
# nhd_sites = nhd_sites.drop(columns='index')

In [ ]:
# mdata_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/'
# with open(os.path.join(mdata_path, 'usgs_gauges_cleaned.json'), 'r') as file:
#     ids = json.load(file)

In [ ]:
# nhd_sites = nhd_sites[nhd_sites['site_id'].isin(ids)]

In [ ]:
nhd_sites = nhd_sites[['site_id', 'name', 'geometry']]
nhd_sites = nhd_sites.to_crs('EPSG:4326')
print(len(nhd_sites['site_id'].unique()))

#### Join tiles to USGS gauges to make reference table

In [ ]:
data_path = '/nas/cee-ice/data/SWOT/tiles/swot_science_coverage_merged.parquet'
# Read in simplified SWOT science tiles
sci_tiles = gpd.read_parquet(path=data_path)

In [ ]:
# # Project CRS to WGS 84 / Pseudo-Mercator
# sci_tiles_wm = sci_tiles.to_crs(epsg=3857)

In [ ]:
# # Get bounding box of gauges
# bounds_gauges = nhd_sites.total_bounds
# # Clip SWOT science tiles to bounding box
# sci_tiles_clip = gpd.clip(gdf=sci_tiles, mask=bounds_gauges)

In [ ]:
# Join the SWOT tiles and HUC4 boundaries by intersection
joined = gpd.sjoin(sci_tiles, nhd_sites, how='inner', predicate='intersects')

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))

joined.plot(alpha=0.5, ax=ax)
nhd_sites.plot(ax=ax, color='k', markersize=2)

In [ ]:
joined = joined.reset_index().rename(columns={'index': 'index_old'})

cleaned = joined[['Name', 'site_id']].sort_values(by=['Name', 'site_id']).rename(columns={'Name': 'tile'}).reset_index().drop(columns='index')

cleaned.to_csv('../data/usgs_gauges_swot_science_tiles.csv', index=False)